In [ ]:
# Cell 1: Imports
import re
import shutil
import random
import csv
from pathlib import Path
from collections import defaultdict, Counter

print('Imports OK')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# Training source: UPDATED_WS2025 IR24 (triplicates, DJI filenames)
ROOT         = "/path/to/folder"
OUT          = "/path/to/folder"

SEVERITIES = ["Control", "Medium", "High"]
DAI        = ["07DAI", "11DAI", "15DAI", "18DAI", "27DAI", "34DAI"] # Change this to proper time periods, this is for the Dry Season
# DAI        = ["08DAI", "12DAI", "15DAI", "19DAI", "28DAI", "34DAI"] # This is the time period for the Wet Season
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT        = (0.80, 0.15, 0.05)   # train / val / test
SEED         = 42

CLASSES = [f"{s}_{t}" for s in SEVERITIES for t in TIME_PERIODS]
random.seed(SEED)

print(f'Source  : {ROOT}')
print(f'Output  : {OUT}')
print(f'Model   : {MODEL_PT}')
print(f'Classes ({len(CLASSES)}): {CLASSES}')
print(f'Split   : {int(SPLIT[0]*100)}/{int(SPLIT[1]*100)}/{int(SPLIT[2]*100)}')

In [ ]:
# Cell 3: Scan - group by (severity, plant_id) across ALL time periods
# IMPORTANT: the leading number is a PLOT/PLANT id, not an image counter.
# The SAME plant (e.g. 289) appears in Control_T1..T4. To avoid plot-identity
# leakage, the split decision must be made per plant, spanning all time periods.
# Unique-plant key = (severity, plant_id) - a Control plot stays Control over time.
PATTERN = re.compile(r'^(\d+)_')

# (severity, plant_id) -> list of (class_key, Path) across all time periods
plant_images = defaultdict(list)
skipped = []

for severity in SEVERITIES:
    sev_dir = ROOT / severity
    if not sev_dir.exists():
        print(f'[WARN] Not found: {sev_dir}')
        continue

    for time_dir in sorted(sev_dir.iterdir()):
        if not time_dir.is_dir():
            continue

        folder_time = time_dir.name.strip().upper()   # e.g. T1
        if folder_time not in TIME_PERIODS:
            continue

        for f in sorted(time_dir.iterdir()):
            if f.suffix.lower() not in IMAGE_EXTS:
                continue

            m = PATTERN.match(f.stem)
            if not m:
                skipped.append(str(f))
                continue

            plant_id  = m.group(1)
            class_key = (severity, folder_time)
            plant_images[(severity, plant_id)].append((class_key, f))

n_plants = len(plant_images)
n_imgs   = sum(len(v) for v in plant_images.values())

print(f'Unique plants (severity, plant_id): {n_plants}')
print(f'Total images                      : {n_imgs}')

print('\nPer-severity:')
for sev in SEVERITIES:
    ps = [k for k in plant_images if k[0] == sev]
    print(f'  {sev:<8}: {len(ps)} plants, {sum(len(plant_images[k]) for k in ps)} images')

if skipped:
    print(f'\n[WARN] {len(skipped)} files skipped (no leading number):')
    for p in skipped[:5]:
        print(f'  {p}')

In [ ]:
# Cell 4: Split by PLANT (all time periods of a plant stay together)
# NO DATA LEAKAGE: a plant and ALL its images go entirely into one split.
train_imgs = defaultdict(list)   # class_key -> [Path,...]
val_imgs   = defaultdict(list)
test_imgs  = defaultdict(list)

split_of_plant = {}   # (severity, plant_id) -> 'train'/'val'/'test'

# Split plants WITHIN each severity so all 12 classes stay balanced
for sev in SEVERITIES:
    plant_keys = sorted([k for k in plant_images if k[0] == sev], key=lambda x: int(x[1]))
    random.shuffle(plant_keys)

    n     = len(plant_keys)
    n_tr  = int(round(n * SPLIT[0]))
    n_val = int(round(n * SPLIT[1]))

    for i, pk in enumerate(plant_keys):
        if   i < n_tr:          split = 'train'
        elif i < n_tr + n_val:  split = 'val'
        else:                   split = 'test'
        split_of_plant[pk] = split

        bucket = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}[split]
        for class_key, f in plant_images[pk]:
            bucket[class_key].append(f)

def count(b):
    return sum(len(v) for v in b.values())

print(f"{'Split':<8}{'Images':>8}")
print(f"{'Train':<8}{count(train_imgs):>8}")
print(f"{'Val':<8}{count(val_imgs):>8}")
print(f"{'Test':<8}{count(test_imgs):>8}")
print(f"{'TOTAL':<8}{count(train_imgs)+count(val_imgs)+count(test_imgs):>8}")

plant_split_count = Counter(split_of_plant.values())
print(f'\nPlants per split: {dict(plant_split_count)}')

print('\nPer-class breakdown:')
for sev in SEVERITIES:
    for tp in TIME_PERIODS:
        key = (sev, tp)
        print(f'  {sev}_{tp:<4} -> train: {len(train_imgs[key]):>3}  '
              f'val: {len(val_imgs[key]):>3}  test: {len(test_imgs[key]):>3}')

In [ ]:
# Cell 4b: Leakage audit - prove no plant crosses splits
plant_to_splits = defaultdict(set)
for pk, sp in split_of_plant.items():
    plant_to_splits[pk].add(sp)

leaks = {pk: sps for pk, sps in plant_to_splits.items() if len(sps) > 1}

if leaks:
    print(f'[FAIL] LEAKAGE: {len(leaks)} plants appear in multiple splits!')
    for pk, sps in list(leaks.items())[:10]:
        print(f'  plant {pk} -> {sps}')
else:
    print('[OK] No plant-identity leakage - each plant is in exactly one split')

assert not leaks, 'Plant-identity leakage detected - fix split logic.'
print('[OK] Assertion passed: every plant and all the DAI images share one split')

In [ ]:
# Cell 5: Write split to disk
from pathlib import Path
OUT = Path(OUT)   # ensure Path (guards against accidental str reassignment)

if OUT.exists():
    print(f'[INFO] Removing existing output: {OUT}')
    shutil.rmtree(OUT)

for split_name, bucket in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
    for class_key, files in bucket.items():
        sev, tp   = class_key
        cls_label = f'{sev}_{tp}'
        dst_dir   = OUT / split_name / cls_label
        dst_dir.mkdir(parents=True, exist_ok=True)

        for counter, src in enumerate(sorted(files), start=1):
            new_name = f'{cls_label}_{counter}{src.suffix}'
            shutil.copy2(src, dst_dir / new_name)
    print(f'OK {split_name}')

print(f'\nDataset written to: {OUT}')

In [ ]:
# Cell 6: Verify split structure
print('Split structure verification:')
for split_name in ['train', 'val', 'test']:
    total = 0
    print(f'\n  {split_name}/')
    for sev in SEVERITIES:
        for tp in TIME_PERIODS:
            cls_dir = OUT / split_name / f'{sev}_{tp}'
            n = len(list(cls_dir.glob('*'))) if cls_dir.exists() else 0
            total += n
            print(f'    {sev}_{tp:<4} -> {n} images')
    print(f'  subtotal: {total}')

print('\nVerification complete')